In [1]:
"""
Assignment 1: Reinforcement Learning from Human Feedback (RLHF)
Notebook: 04_ppo_hh.ipynb

Purpose:
This notebook performs the RLHF policy-optimisation stage using Proximal Policy
Optimisation (PPO). It starts from the SFT model and uses the reward model trained
on Anthropic HH-RLHF to guide policy updates toward preferred responses.

Summary of sections:
1. Environment setup and package imports
   - Imports PPO / RLHF libraries and prepares the runtime environment.

2. Loading models and tokenizer
   - Loads the SFT model, tokenizer, and the HH reward model.

3. PPO dataset preparation
   - Loads and formats prompts used during PPO training.

4. PPO configuration
   - Defines PPO hyperparameters, generation settings, batch sizes, and optimisation settings.

5. PPO training loop
   - Generates responses from the policy model, scores them with the HH reward model,
     and updates the policy using PPO.

6. Logging and monitoring
   - Tracks rewards, training behaviour, and sample outputs during optimisation.

7. Saving outputs
   - Saves the PPO-trained model/checkpoint for later evaluation.

8. Sample generation / inspection
   - Generates example outputs from the PPO-HH model for qualitative review.

Main output:
- PPO-aligned model trained using the HH reward model.
"""

'\nAssignment 1: Reinforcement Learning from Human Feedback (RLHF)\nNotebook: 04_ppo_hh.ipynb\n\nPurpose:\nThis notebook performs the RLHF policy-optimisation stage using Proximal Policy\nOptimisation (PPO). It starts from the SFT model and uses the reward model trained\non Anthropic HH-RLHF to guide policy updates toward preferred responses.\n\nSummary of sections:\n1. Environment setup and package imports\n   - Imports PPO / RLHF libraries and prepares the runtime environment.\n\n2. Loading models and tokenizer\n   - Loads the SFT model, tokenizer, and the HH reward model.\n\n3. PPO dataset preparation\n   - Loads and formats prompts used during PPO training.\n\n4. PPO configuration\n   - Defines PPO hyperparameters, generation settings, batch sizes, and optimisation settings.\n\n5. PPO training loop\n   - Generates responses from the policy model, scores them with the HH reward model,\n     and updates the policy using PPO.\n\n6. Logging and monitoring\n   - Tracks rewards, training

In [2]:
# Install packages

!pip -q install -U \
  "transformers==4.45.2" \
  "datasets==2.21.0" \
  "trl==0.11.4" \
  "peft==0.12.0" \
  "accelerate==0.34.2" \
  "evaluate==0.4.3" \
  "rouge_score" \
  "bert_score" \
  "scikit-learn"

In [8]:
# Import Libraries

import os
import gc
import json
import random
import warnings
import inspect
import shutil
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)
from peft import PeftModel, LoraConfig
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer

warnings.filterwarnings("ignore")


# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')


# Version Check

import peft
import transformers
import trl

print("peft version:", peft.__version__)
print("transformers version:", transformers.__version__)
print("trl version:", trl.__version__)


# Model and Configuration

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
SFT_ADAPTER_PATH = "/content/drive/MyDrive/rlhf_assignment_outputs/sft_qwen25_05b_base"

RM_BASE_MODEL_NAME = "distilbert-base-uncased"
RM_HH_PATH = "/content/drive/MyDrive/rlhf_assignment_outputs/rm_hh"

OUTPUT_DIR = "./outputs/ppo_hh"

SEED = 42
MAX_PROMPT_LENGTH = 256
MAX_NEW_TOKENS = 96
MAX_RM_LENGTH = 512
TRAIN_SAMPLES = 500
BATCH_SIZE = 4
MINI_BATCH_SIZE = 2
GRAD_ACCUM = 1
PPO_EPOCHS = 2
LEARNING_RATE = 1e-5

os.makedirs("./outputs", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print("Device:", device)
print("Base model:", BASE_MODEL_NAME)
print("SFT adapter:", SFT_ADAPTER_PATH)
print("Reward model:", RM_HH_PATH)
print("SFT exists:", os.path.exists(SFT_ADAPTER_PATH))
print("RM exists:", os.path.exists(RM_HH_PATH))


# Show Saved Files

if os.path.exists(SFT_ADAPTER_PATH):
    print("\nFiles in SFT adapter path:")
    for f in os.listdir(SFT_ADAPTER_PATH):
        print(" ", f)

if os.path.exists(RM_HH_PATH):
    print("\nFiles in RM_HH path:")
    for f in os.listdir(RM_HH_PATH):
        print(" ", f)


# Helper Function: sanitize adapter config for current PEFT version

def sanitize_lora_config(config_path):
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"adapter_config.json not found at: {config_path}")

    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    allowed_keys = set(inspect.signature(LoraConfig.__init__).parameters.keys())
    allowed_keys.discard("self")

    removed = {}
    clean_cfg = {}

    for k, v in cfg.items():
        if k in allowed_keys:
            clean_cfg[k] = v
        else:
            removed[k] = v

    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(clean_cfg, f, indent=2)

    return removed, clean_cfg


# Sanitize SFT adapter_config.json

sft_config_path = os.path.join(SFT_ADAPTER_PATH, "adapter_config.json")
removed_sft, clean_sft = sanitize_lora_config(sft_config_path)

print("\nSanitized SFT adapter_config.json")
print("Removed unsupported SFT keys:", list(removed_sft.keys()))
print("Remaining SFT keys:", list(clean_sft.keys()))


# Load tokenizer and PPO model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto"
)

sft_peft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_PATH)

ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(sft_peft_model)
ppo_model = ppo_model.to(device)

ref_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto"
)

ref_peft_model = PeftModel.from_pretrained(ref_base_model, SFT_ADAPTER_PATH)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(ref_peft_model)
ref_model = ref_model.to(device)

print("\nPPO policy and reference model loaded successfully.")


# Sanitize RM-HH adapter_config.json

rm_config_path = os.path.join(RM_HH_PATH, "adapter_config.json")
removed_rm, clean_rm = sanitize_lora_config(rm_config_path)

print("\nSanitized RM adapter_config.json")
print("Removed unsupported RM keys:", list(removed_rm.keys()))
print("Remaining RM keys:", list(clean_rm.keys()))


# Load reward model

rm_tokenizer = AutoTokenizer.from_pretrained(RM_BASE_MODEL_NAME, use_fast=True)

if rm_tokenizer.pad_token is None:
    if rm_tokenizer.sep_token is not None:
        rm_tokenizer.pad_token = rm_tokenizer.sep_token
    elif rm_tokenizer.eos_token is not None:
        rm_tokenizer.pad_token = rm_tokenizer.eos_token
    else:
        rm_tokenizer.add_special_tokens({"pad_token": "[PAD]"})

rm_base_model = AutoModelForSequenceClassification.from_pretrained(
    RM_BASE_MODEL_NAME,
    num_labels=1
).to(device)

reward_model = PeftModel.from_pretrained(rm_base_model, RM_HH_PATH)
reward_model = reward_model.to(device)
reward_model.eval()

print("Reward model loaded successfully.")


# Load HH-RLHF prompts

dataset = load_dataset("Anthropic/hh-rlhf", split="train")
dataset = dataset.shuffle(seed=SEED).select(range(TRAIN_SAMPLES))

print("\nLoaded HH train subset:", len(dataset))
print("Columns:", dataset.column_names)
print(dataset[0])


# Helper Functions

def extract_prompt_from_hh(example):
    text = str(example["chosen"])
    marker = "\n\nAssistant:"
    idx = text.rfind(marker)

    if idx != -1:
        human_part = text[:idx].strip()
    else:
        human_part = text.strip()

    human_part = human_part.replace("\n\nHuman:", "").strip()
    prompt = f"Instruction: {human_part}\n\nResponse:"
    return {"prompt": prompt}

def tokenize_prompt(example):
    encoded = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
        padding=False
    )
    return {
        "query": example["prompt"],
        "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long)
    }

def collator(features):
    return {k: [f[k] for f in features] for k in features[0]}

def build_full_text(prompt, response):
    prompt = str(prompt).strip()
    response = str(response).strip()
    if prompt.endswith("Assistant:"):
        return f"{prompt} {response}"
    return f"{prompt}\n{response}"

def score_texts(texts):
    inputs = rm_tokenizer(
        texts,
        truncation=True,
        max_length=MAX_RM_LENGTH,
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = reward_model(**inputs).logits.squeeze(-1)

    return logits.detach().cpu()

dataset = dataset.map(extract_prompt_from_hh)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "prompt"])
dataset = dataset.filter(lambda x: len(x["prompt"]) > 0)
dataset = dataset.map(tokenize_prompt)

print("\nProcessed PPO dataset size:", len(dataset))
print("Example prompt:")
print(dataset[0]["query"][:500])


# PPO Configuration

ppo_config = PPOConfig(
    model_name=BASE_MODEL_NAME,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    mini_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    ppo_epochs=PPO_EPOCHS,
    seed=SEED,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=dataset,
    data_collator=collator
)

generation_kwargs = {
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": True,
    "top_p": 0.9,
    "temperature": 0.7,
    "repetition_penalty": 1.1,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

print("\nPPOTrainer created successfully.")


# PPO Training Loop

training_logs = []
sample_outputs = []

for step, batch in enumerate(tqdm(ppo_trainer.dataloader, desc="PPO HH training")):
    query_tensors = batch["input_ids"]
    query_texts = batch["query"]

    query_tensors = [
        q if isinstance(q, torch.Tensor) else torch.tensor(q, dtype=torch.long)
        for q in query_tensors
    ]

    response_tensors = ppo_trainer.generate(
        query_tensors,
        return_prompt=False,
        **generation_kwargs
    )

    response_texts = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)

    full_texts = [build_full_text(q, r) for q, r in zip(query_texts, response_texts)]
    reward_scores = score_texts(full_texts)

    rewards = [torch.tensor(float(x)) for x in reward_scores]

    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

    mean_reward = float(torch.mean(reward_scores).item())
    std_reward = float(torch.std(reward_scores).item()) if len(reward_scores) > 1 else 0.0
    mean_len = float(np.mean([len(tokenizer.encode(x, add_special_tokens=False)) for x in response_texts]))

    row = {
        "step": step,
        "mean_reward": mean_reward,
        "std_reward": std_reward,
        "mean_response_len": mean_len
    }

    for k, v in stats.items():
        try:
            row[k] = float(v) if not isinstance(v, (list, dict, tuple)) else str(v)
        except Exception:
            row[k] = str(v)

    training_logs.append(row)

    if step % 10 == 0:
        print("=" * 80)
        print("Step:", step)
        print("Mean reward:", mean_reward)
        print("Sample prompt:")
        print(query_texts[0][:500])
        print("Sample response:")
        print(response_texts[0][:500])

        sample_outputs.append({
            "step": step,
            "prompt": query_texts[0],
            "response": response_texts[0],
            "reward": float(reward_scores[0].item())
        })

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Save PPO model and logs

ppo_trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

pd.DataFrame(training_logs).to_csv(f"{OUTPUT_DIR}/ppo_hh_logs.csv", index=False)

with open(f"{OUTPUT_DIR}/ppo_hh_samples.jsonl", "w", encoding="utf-8") as f:
    for item in sample_outputs:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("\nSaved PPO-HH model to:", OUTPUT_DIR)


# Copy PPO model to Google Drive

drive_output_root = "/content/drive/MyDrive/rlhf_assignment_outputs"
os.makedirs(drive_output_root, exist_ok=True)

drive_ppo_path = os.path.join(drive_output_root, "ppo_hh")

if os.path.exists(drive_ppo_path):
    shutil.rmtree(drive_ppo_path)

shutil.copytree(OUTPUT_DIR, drive_ppo_path)

print("Copied PPO-HH model to Drive:", drive_ppo_path)
print("Files in Drive PPO-HH folder:")
for f in os.listdir(drive_ppo_path):
    print(" ", f)


# Final sanity generation

test_prompt = "Instruction: Explain why human feedback is useful in RLHF.\n\nResponse:"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = ppo_trainer.model.pretrained_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

print("=" * 80)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
peft version: 0.12.0
transformers version: 4.45.2
trl version: 0.11.4
Device: cuda
Base model: Qwen/Qwen2.5-0.5B-Instruct
SFT adapter: /content/drive/MyDrive/rlhf_assignment_outputs/sft_qwen25_05b_base
Reward model: /content/drive/MyDrive/rlhf_assignment_outputs/rm_hh
SFT exists: True
RM exists: True

Files in SFT adapter path:
  checkpoint-100
  checkpoint-125
  README.md
  adapter_model.safetensors
  tokenizer.json
  training_args.bin
  chat_template.jinja
  tokenizer_config.json
  adapter_config.json

Files in RM_HH path:
  checkpoint-50
  checkpoint-58
  tokenizer_config.json
  adapter_config.json
  training_args.bin
  adapter_model.safetensors
  tokenizer.json
  README.md

Sanitized SFT adapter_config.json
Removed unsupported SFT keys: []
Remaining SFT keys: ['alpha_pattern', 'auto_mapping', 'base_model_name_or_path', 'bias', 'fan_in_fan_out', 'inference

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Reward model loaded successfully.

Loaded HH train subset: 500
Columns: ['chosen', 'rejected']
{'chosen': '\n\nHuman: Why did cells originally combine together to create life?\n\nAssistant: Because their simple components -- chemicals -- interacted in particular ways.  And because of chemical processes involving acids and bases, certain kinds of chemicals can begin to self-organize into larger structures, like membrane-bounded compartments.  And it’s from those compartments that life eventually emerged.', 'rejected': '\n\nHuman: Why did cells originally combine together to create life?\n\nAssistant: Cells combine because they benefit from cooperation, since they can have less competition for resources by working together.'}


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Processed PPO dataset size: 500
Example prompt:
Instruction: Human: Why did cells originally combine together to create life?

Response:

PPOTrainer created successfully.


PPO HH training:   0%|          | 0/125 [00:00<?, ?it/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Step: 0
Mean reward: -0.026954613626003265
Sample prompt:
Instruction: Human: I'm tired of sweeping my house. What can I do?

Response:
Sample response:
 You could try taking a shower to relax and refresh your energy, or you might want to take some yoga to get your body moving and feel more energized. You could also try reading a book or doing something that interests you to pass the time while you clean.

Options:
A). Showering;
B). Reading books;
C). Cleaning your house;
D). None of the above;

The answer is C). Cleaning your house. This response provides several options for what someone might
Step: 10
Mean reward: -0.0007310695946216583
Sample prompt:
Instruction: Human: I am taking a business trip to  Ostrava, in the Czech Republic  next month. I will have a free day during this trip, and I want to see some of the local site and scenery. Can you create an itinerary for that day that includes few different sites?

Assistant: Sure, I can help you plan a great itinerary for a day trip